Ссылка на урок: [Спринт 2/6: 2 RAG: от сырых данных до точных ответов → Тема 2/4: Векторные базы и индексы → Урок 3/5: FAISS: индексы, скорость и компромиссы]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/7c1be2cd-4a52-4f8a-8dce-db7d2e89a79d/lessons/5d6f1006-544b-49f4-af26-38a50457aeeb/#6f3d0d47-7b3e-45ce-9c05-5547bc720094 )
Часть 1

In [1]:
import numpy as np                                          # Матрицы эмбеддингов
from datasets import load_dataset                           # Скачивание корпуса MIRACL
from sentence_transformers import SentenceTransformer       # Эмбеддер на GPU

# --- Корпус: стримим русский MIRACL, берём пассажи разумной длины ---
# streaming=True не качает все 9.5 млн пассажей, а отдаёт их по одному
stream = load_dataset(
    "json",
    data_files="hf://datasets/miracl/miracl-corpus/"
               "miracl-corpus-v1.0-ru/docs-*.jsonl.gz",
    split="train", streaming=True)
texts: list[str] = []
for row in stream:
    t = row["text"].strip()
    if 100 <= len(t) <= 1000:          # Отсекаем слишком короткие и длинные
        texts.append(t)
    if len(texts) >= 100_100:
        break

CORPUS = texts[:100_000]               # 100 000 пассажей — корпус
QUERIES = texts[100_000:100_100]       # 100 пассажей — запросы
assert len(CORPUS) == 100_000 and len(QUERIES) == 100


def embed_e5_docs(model, texts):
    """Эмбеддинги документов через multilingual-e5: к каждому
    документу спереди добавляется префикс 'passage: '."""
    inputs = ["passage: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True,
                        batch_size=64, show_progress_bar=True)


def embed_e5_queries(model, texts):
    """Эмбеддинги запросов через multilingual-e5: к каждому запросу
    спереди добавляется префикс 'query: '."""
    inputs = ["query: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True,
                        batch_size=64, show_progress_bar=True)


# --- Считаем эмбеддинги на GPU (это самый долгий шаг, около 10–15 минут) ---
# FAISS требует непрерывный массив float32.
e5 = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")
corpus_vecs = np.ascontiguousarray(embed_e5_docs(e5, CORPUS), dtype="float32")
query_vecs = np.ascontiguousarray(embed_e5_queries(e5, QUERIES), dtype="float32")

# Сохраняем, чтобы не пересчитывать в Задании 6
np.save("corpus_vecs.npy", corpus_vecs)
np.save("query_vecs.npy", query_vecs)
d = corpus_vecs.shape[1]               # Размерность вектора e5-base: 768
print(f"Корпус: {corpus_vecs.shape}, запросы: {query_vecs.shape}")

/home/ubuntu/rag-vector-db/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 2/2 [00:00<00:00,  2.04it/s]


Корпус: (100000, 768), запросы: (100, 768)


In [6]:
import time
import numpy as np
import faiss

corpus_vecs = np.load("corpus_vecs.npy")[:10_000]    # Срез на 10 000
query_vecs = np.load("query_vecs.npy")[:100]          # 10 запросов
d = corpus_vecs.shape[1]

# 1. Точный индекс — эталон, без обучения
flat = faiss.IndexFlatIP(d)
flat.add(corpus_vecs)

# 2. IVFFlat: разбиваем пространство на ячейки
nlist = 100                            # ~ √10000, по ячейке хватит данных на k-means
quantizer = faiss.IndexFlatIP(d)       # По нему IVF выбирает ближайшие ячейки
ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
ivf.train(corpus_vecs)                 # Обучаем k-means: находим центроиды
ivf.add(corpus_vecs)
ivf.nprobe = 16


def search_time(index, queries, k=5):
    """Время поиска top-k для всех запросов, в секундах."""
    t0 = time.perf_counter()
    index.search(queries, k)
    return time.perf_counter() - t0


# Сравним выдачу: какая доля точной top-5 нашлась у IVF
_, flat_ids = flat.search(query_vecs, 5)
_, ivf_ids = ivf.search(query_vecs, 5)
overlap = np.mean([len(set(a) & set(b)) / 5
                   for a, b in zip(flat_ids, ivf_ids)])

print(f"{'Индекс':<16}{'Поиск 10 зпр., с':>20}")
print(f"{'IndexFlatIP':<16}{search_time(flat, query_vecs):>20.5f}")
print(f"{'IndexIVFFlat':<16}{search_time(ivf, query_vecs):>20.5f}")
print(f"\nСовпадение top-5 IVF с точным поиском: {overlap:.2f}")

Индекс              Поиск 10 зпр., с
IndexFlatIP                  0.04833
IndexIVFFlat                 0.00839

Совпадение top-5 IVF с точным поиском: 0.93


In [7]:
import os
import time
import numpy as np
import faiss

corpus_full = np.load("corpus_vecs.npy")     # (100000, 768)
query_vecs = np.load("query_vecs.npy")       # (100, 768)
d = corpus_full.shape[1]
K = 10


def build_flat(vecs):
    """Точный индекс — эталон. Обучение не нужно."""
    index = faiss.IndexFlatIP(d)
    index.add(vecs)
    return index


def build_ivf(vecs, nlist, nprobe):
    """IVFFlat: nlist ячеек, проверяем nprobe ближайших при запросе."""
    quantizer = faiss.IndexFlatIP(d)
    # Пропуск 1: создайте IndexIVFFlat(quantizer, d, nlist,
    # METRIC_INNER_PRODUCT), обучите на vecs и добавьте векторы
    index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
    index.train(vecs)                        # Учим k-means: центроиды ячеек
    index.add(vecs)
    index.nprobe = nprobe
    return index


def build_hnsw(vecs, M=32, ef_construction=200, ef_search=64):
    """HNSW: граф с M рёбрами на вершину. Обучение не нужно."""
    # Пропуск 2: создайте IndexHNSWFlat(d, M, METRIC_INNER_PRODUCT),
    # задайте index.hnsw.efConstruction, добавьте векторы,
    # затем задайте index.hnsw.efSearch
    index = faiss.IndexHNSWFlat(d, M, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = ef_construction   # Тщательность сборки графа
    index.add(vecs)
    index.hnsw.efSearch = ef_search               # Ширина обхода при поиске
    return index


def index_size_mb(index, path):
    """Пишет индекс на диск и возвращает размер файла в мегабайтах."""
    faiss.write_index(index, path)
    return os.path.getsize(path) / 1e6


def search_time(index, queries, k=K):
    """Время поиска top-k для всех запросов, в секундах."""
    index.search(queries, k)                 # Прогрев, чтобы не мерить разовые накладные
    t0 = time.perf_counter()
    index.search(queries, k)
    return time.perf_counter() - t0


def recall_at_k(approx_index, exact_index, queries, k=K):
    """Доля top-k точного поиска (эталон), которую воспроизводит
    приближённый индекс. Усреднение по всем запросам."""
    _, exact = exact_index.search(queries, k)
    _, approx = approx_index.search(queries, k)
    hits = 0
    for exact_row, approx_row in zip(exact, approx):
        # Пропуск 3: пересечение эталонной и приближённой top-k
        common = set(exact_row) & set(approx_row)
        hits += len(common)
    return hits / (len(queries) * k)


print(f"{'Размер':>8}{'Индекс':<16}{'Build, с':>10}"
      f"{'Размер, МБ':>12}{'Поиск, с':>10}{'recall@10':>12}")
for n in [10_000, 100_000]:
    vecs = np.ascontiguousarray(corpus_full[:n])
    nlist = 100 if n == 10_000 else 1024     # ~ √N, хватает данных на k-means

    builders = [
        ("IndexFlat", lambda v: build_flat(v)),
        ("IVFFlat", lambda v: build_ivf(v, nlist, nprobe=16)),
        ("HNSW", lambda v: build_hnsw(v)),
    ]
    flat_ref = build_flat(vecs)              # Эталон для recall на этом размере
    for name, build in builders:
        t0 = time.perf_counter()
        index = build(vecs)
        build_s = time.perf_counter() - t0
        size = index_size_mb(index, f"{name}_{n}.faiss")
        secs = search_time(index, query_vecs)
        rec = recall_at_k(index, flat_ref, query_vecs)
        print(f"{n:>8}{name:<16}{build_s:>10.2f}"
              f"{size:>12.2f}{secs:>10.5f}{rec:>12.3f}")

  РазмерИндекс            Build, с  Размер, МБ  Поиск, с   recall@10
   10000IndexFlat             0.01       30.72   0.03839       1.000
   10000IVFFlat               0.56       31.11   0.00722       0.918
   10000HNSW                  2.68       33.44   0.00774       0.970
  100000IndexFlat             0.14      307.20   0.53113       1.000
  100000IVFFlat              45.21      311.15   0.01460       0.751
  100000HNSW                 47.62      334.42   0.01687       0.958
